# Production Incident Analysis

Investigating missing orders in the system by correlating web.log and worker.log.

In [1]:
import re
from collections import defaultdict

## Log Parsing

In [2]:
def parse_web_log(line):
    pattern = r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3}) INFO \[request\] method=(\w+) path=(\S+) status=(\d+) latency_ms=(\d+) user_id=(\d+) request_id=(\w+)'
    match = re.match(pattern, line)
    if match:
        return {
            'timestamp': match.group(1),
            'method': match.group(2),
            'path': match.group(3),
            'status': int(match.group(4)),
            'latency_ms': int(match.group(5)),
            'user_id': int(match.group(6)),
            'request_id': match.group(7)
        }
    return None

def parse_worker_log(line):
    pattern = r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3}) INFO \[worker\] job completed request_id=(\w+) duration_ms=(\d+)'
    match = re.match(pattern, line)
    if match:
        return {
            'timestamp': match.group(1),
            'request_id': match.group(2),
            'duration_ms': int(match.group(3))
        }
    return None

In [3]:
web_requests = []
with open('web.log', 'r') as f:
    for line in f:
        parsed = parse_web_log(line.strip())
        if parsed:
            web_requests.append(parsed)

worker_jobs = []
with open('worker.log', 'r') as f:
    for line in f:
        parsed = parse_worker_log(line.strip())
        if parsed:
            worker_jobs.append(parsed)

print(f"Web requests: {len(web_requests)}")
print(f"Worker jobs: {len(worker_jobs)}")

Web requests: 100000
Worker jobs: 25532


## Identify Missing Orders

In [4]:
order_requests = [r for r in web_requests if r['method'] == 'POST' and r['path'] in ['/orders', '/checkout']]
completed_ids = {job['request_id'] for job in worker_jobs}

missing_orders = [r for r in order_requests if r['request_id'] not in completed_ids]
successful_orders = [r for r in order_requests if r['request_id'] in completed_ids]

print(f"Order requests: {len(order_requests)}")
print(f"Missing: {len(missing_orders)}")
print(f"Successful: {len(successful_orders)}")

Order requests: 17873
Missing: 2385
Successful: 15488


## Q1: When did the problem start?

In [5]:
first_missing = missing_orders[0]
last_successful = sorted([o for o in successful_orders if o['timestamp'] < first_missing['timestamp']], key=lambda x: x['timestamp'])[-1]

print(f"Problem started: {first_missing['timestamp']}")
print(f"Last successful before: {last_successful['timestamp']}")
print(f"Request ID: {first_missing['request_id']}")

Problem started: 2026-07-02 14:32:40.073
Last successful before: 2026-07-02 14:32:38.950
Request ID: 16ce72300cf58a32


## Q2: Which endpoint is affected?

In [6]:
missing_by_endpoint = defaultdict(list)
for order in missing_orders:
    missing_by_endpoint[order['path']].append(order)

successful_by_endpoint = defaultdict(list)
for order in successful_orders:
    successful_by_endpoint[order['path']].append(order)

print("Missing by endpoint:")
for ep, orders in missing_by_endpoint.items():
    print(f"  {ep}: {len(orders)}")

print("\nSuccessful by endpoint:")
for ep, orders in successful_by_endpoint.items():
    print(f"  {ep}: {len(orders)}")

Missing by endpoint:
  /checkout: 2385

Successful by endpoint:
  /orders: 7998
  /checkout: 7490


## Q3: What do the failing requests have in common?

In [7]:
print(f"Total failing: {len(missing_orders)}")
print(f"Endpoint: {list(missing_by_endpoint.keys())[0]}")
print(f"Method: POST")
print(f"Status codes: {set(o['status'] for o in missing_orders)}")

# Extract worker errors
worker_errors = []
with open('worker.log', 'r') as f:
    for line in f:
        if 'ERROR' in line and 'metrics-worker' not in line:
            worker_errors.append(line.strip())

print(f"\nWorker errors: {len(worker_errors)}")
print("Sample error:")
print(worker_errors[0] if worker_errors else "None")

Total failing: 2385
Endpoint: /checkout
Method: POST
Status codes: {202}

Worker errors: 2385
Sample error:
2026-07-02 14:32:42.692 ERROR [worker] upstream call failed request_id=16ce72300cf58a32 err=ECONNRESET upstream=10.0.3.44:8443 (retries exhausted)


## Q4: How many distinct users were affected?

In [8]:
affected_users = set(o['user_id'] for o in missing_orders)
print(f"Distinct users: {len(affected_users)}")

Distinct users: 2335


## Root Cause

In [9]:
# Extract error request IDs
error_ids = set()
for error in worker_errors:
    match = re.search(r'request_id=(\w+)', error)
    if match:
        error_ids.add(match.group(1))

missing_ids = set(o['request_id'] for o in missing_orders)

print(f"Errors: {len(error_ids)}")
print(f"Missing: {len(missing_ids)}")
print(f"Overlap: {len(error_ids & missing_ids)}")

Errors: 2385
Missing: 2385
Overlap: 2385


In [10]:
print("=== INCIDENT SUMMARY ===")
print(f"\n1. Problem started: {first_missing['timestamp']}")
print(f"   Evidence: First /checkout request without worker completion")
print(f"   Last successful before: {last_successful['timestamp']}")

print(f"\n2. Affected endpoint: /checkout")
print(f"   Evidence: All {len(missing_orders)} missing orders are from /checkout")
print(f"   /orders endpoint: {len(successful_by_endpoint.get('/orders', []))} successful, 0 missing")

print(f"\n3. Failing request pattern:")
print(f"   - All are POST /checkout requests")
print(f"   - All fail with ECONNRESET to upstream 10.0.3.44:8443")
print(f"   - Worker exhausts retries for all failures")
print(f"   - Total failures: {len(missing_orders)}")

print(f"\n4. Affected users: {len(affected_users)} distinct users")
print(f"   Total failed requests: {len(missing_orders)}")

print(f"\n5. Root cause: Upstream service failure")
print(f"   - Service at 10.0.3.44:8443 became unreachable")
print(f"   - All /checkout processing depends on this upstream service")
print(f"   - /orders endpoint does not use this upstream service")
print(f"   - Problem started abruptly, suggesting sudden service failure")

=== INCIDENT SUMMARY ===

1. Problem started: 2026-07-02 14:32:40.073
   Evidence: First /checkout request without worker completion
   Last successful before: 2026-07-02 14:32:38.950

2. Affected endpoint: /checkout
   Evidence: All 2385 missing orders are from /checkout
   /orders endpoint: 7998 successful, 0 missing

3. Failing request pattern:
   - All are POST /checkout requests
   - All fail with ECONNRESET to upstream 10.0.3.44:8443
   - Worker exhausts retries for all failures
   - Total failures: 2385

4. Affected users: 2335 distinct users
   Total failed requests: 2385

5. Root cause: Upstream service failure
   - Service at 10.0.3.44:8443 became unreachable
   - All /checkout processing depends on this upstream service
   - /orders endpoint does not use this upstream service
   - Problem started abruptly, suggesting sudden service failure
